## LangChain + OPenAI from scracth

# We will learn to real AI Applications using:

- python
- OpenAI
- LangChain
- Prompt Templates
- Structured outputs
- Memory
- PDF loading
- Text Splittig
- Embedding
- Vector Database
- RAG: Retrival-Augmented Genration

We will build an AI assistant that can read a PDF and answer questions from it.

Example:

> "What problem does LangChain solve?"

> "Explain prompt templates."

> "Why do we split documents?"

This RAG is useful for:

- Course assistants
- HR policy bots
- Legal document assistants
- Resume analyzers
- Finance report readers
- ESG report assistants
- Internal company knowledge bots

# 1. Why LangChain?

OpenAI model alone can answer questions.

But real applications need more than simple Q&A.

Real applications need:

- Prompt Template
- Document Reading
- Memory
- Structured JSON output
- Multi-step workflows
- Search over PDFs
- Retrieval from kowledge bases

LangChain helps us organize all these steps cleanly. 




#Load Environment Variables

We will load the API key from the `.env` file.

This keeps our code clean and secure.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("MODEL", "gpt-3.5-turbo")
embedding_model_name = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

if api_key:
    print("API key has been loaded successfully.")
if model_name:
    print("Model name has been loaded successfully.")
if embedding_model_name:
    print("Embedding model name has been loaded successfully.")

else:
    print("Failed to load API key or model names. Please check your .env file.")    

## OpenAI Without LangChain

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

response = client.responses.create(
    model=model_name,
    input="Explain Langchain in one simple sentence for a beginner."
)

print(response.output_text)

In [ ]:
from langchain_openai import ChatOpenAI

llm =ChatOpenAI(
    model=model_name,
    temperature=0.7
)

response = llm.invoke("Explain Langchain in one simple sentence for a beginner.")

print(response.content)

## Prompt Templates

Problem:

If we write prompts manually again and again, we repeat ourselves.

Example:

Explain Selenium to a beginner.  
Explain Playwright to a beginner.  
Explain LangChain to a beginner.  

Instead, we can create a reusable template.

A prompt template is like a form with blanks.

Example:

"Explain {topic} to a beginner with one real-life example."

This is our first chain.

The flow is:

User Input → Prompt Template → OpenAI Model → Output Parser → Final Answer

The pipe symbol `|` means:

"Send the output of one step to the next step."

This is called LangChain Expression Language (LCEL).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


prompt = ChatPromptTemplate.from_template(
    """
    You are a friendly AI trainer.

    Explain the topic: {topic}

    Rules:
    - Explain in beginner-friendly language
    - Use one real-life example
    - Give one interview-style question at the end
    """
)

chain = prompt | llm | StrOutputParser()

response = chain.invoke({"topic": "Langchain"})

print(response)

## Structured Output

Normal LLM output is plain text.

But real applications often need JSON.

Example:

Instead of this:

"Rakesh is a Senior QA Automation Test Engineer with 10 years of experience."

We want this: JSON style

{
"name": "Rakesh",
"role": "Senior QA Automation Test Engineer"
"experience_years": 10,
"skills": ["Selenium, "Playwright", "Pyhon"]
}

Structured output is useful when we want to:

- Save data into database
- Send data to API
- Show data in UI
- Validate response format

In [ ]:
from typing import List
from pydantic import BaseModel, Field

class StudentProfile(BaseModel):
    name: str = Field(description="Name of the person")
    role: str = Field(description="Current role")
    experience_years: int = Field(description="Total years of experience")
    skills: List[str] = Field(description="List of technical skills")

structured_llm = llm.with_structured_output(StudentProfile)

profile_text = """
Rakesh Kumar Singh is a Senior QA Automation Engineer with 10 years of experience.
He knows Selenium, Java, Playwright, Python, API Testing, and Generative AI.
"""

result = structured_llm.invoke(
    f"Extract the profile information from this text:\n{profile_text}"
)

print(result)
print("Name:", result.name)
print("Skills:", result.skills)

## Load a PDF

Now we move toward a real use case.

We will load a PDF and ask questions from it.

Keep our PDF in the same folder as this notebook.

Example file name:

dotcom-secrets.pdf

## PyPDFLoader reads the PDF.

It converts every page into a Document object.

Each Document has:

- page_content
- metadata

page_content contains the text.

metadata contains information like page number and source file.

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_path = Path("dotcom-secrets.pdf")

if pdf_path.exists():
    print(f"PDF file found at: {pdf_path}")

else: 
    print(f"PDF file not found at: {pdf_path}. Please check the path and try again.")

In [ ]:
loader = PyPDFLoader(str(pdf_path))

documents = loader.load()

print(f"Number of documents loaded: {len(documents)}")
print(f"\nFirst document content:\n{documents[101].page_content[:5000]}")
print(f"\nMetadata of the document:\n{documents[101].metadata}")


## Text Splitting / Chunking

chunk_size=800 means each chunk will have around 800 characters.

chunk_overlap=150 means some text will repeat between chunks.

Why overlap?

Because sometimes an important answer may start in one chunk and continue in the next.

Overlap helps preserve context.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks created: {len(chunks)}")
print(f"\nFirst chunk content:")
print(chunks[440].page_content)
print(f"\nMetadata of the first chunk:")
print(chunks[440].metadata)


print(f"\nWhich page has this content:")
print(f"Page Number: {chunks[440].metadata.get('page_label')}")

## Embeddings

Embeddings convert text into numbers.

But these numbers capture meaning.

Example:

"car" and "vehicle" will be close in meaning.

"car" and "banana" will be far in meaning.

Embeddings help us search by meaning, not only by exact keyword.

Do not worry about the numbers.

The important point is:

Text has been converted into a mathematical representation.

Now we can compare meaning mathematically.

Do not worry about the numbers.

The important point over here is:

### Now we can compare the meaning mathematicaly

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model=embedding_model_name,
)

#this is the text what user has asked we want 
# to convert into a vector using the embedding model so that it sits around the answer in vector database
sample_text = "What is the main topic of the document?"

sample_vector = embeddings.embed_query(sample_text)


print("Vector length:", len(sample_vector))
print("First 5 values of the vector:", sample_vector[:5])

## The Vector 

A vector store is a database for embeddings.

We will use FAISS or we can use ChromaDb.

FAISS helps us search similar chunks quickly.

Flow:

Chunks → Embeddings → FAISS Vector Store

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector store created successfully.")

## Retriever

A Retriever finds the most relevant chunks for a user quesiton.

Example:

User asks:

"What problem does dotCom-seccrets solve?"

Retriever searches the vector store and finds related PDF chunks

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3} #Retrieve the top 3 most relevant chunks. These chunks will be passed to the LLM as context.
    #This reduces hallucination because the model answers based on retrieved content.
)

question = "What are the key strategies mentioned in the document for building a successful online business?"

relevant_chunks = retriever.invoke(question)

print(f"Number of relevant documents retrieved: {len(relevant_chunks)}")

for i, doc in enumerate(relevant_chunks, start=1):
    print(f"\nRelevant Document {i}:")
    print(f"Content: {doc.page_content[:500]}...")
    print(f"Metadata: {doc.metadata}")

#  Build Final PDF Question Answering Chain

Now we combine everything:

PDF → Loader → Documents → Splitter → Chunks → Embeddings → Vector Store → Retriever → Prompt → LLM → Answer

This is called RAG:

Retrieval-Augmented Generation.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rag_prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful AI teaching assistant.

    Answer the user's question using ONLY the given context.

    If the answer is not available in the context, say:
    "I could not find this in the provided PDF."

    Keep the answer simple, practical, and classroom-friendly.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

rag_chain = rag_prompt | llm | StrOutputParser()

def format_docs(docs):
    formatted_text = []
    for i, doc in enumerate(docs, start=1):
        page = doc.metadata.get("page", "unknown")
        source = doc.metadata.get("source", "unknown")
        formatted_text.append(
            f"Source {i} | Page: {page} | File: {source}\n{doc.page_content}"
        )
    return "\n\n".join(formatted_text)

def ask_pdf(question: str):
    relevant_docs = retriever.invoke(question)
    context = format_docs(relevant_docs)

    answer = rag_chain.invoke({
        "context": context,
        "question": question
    })

    print("=" * 80)
    print("QUESTION:")
    print(question)
    print("\nANSWER:")
    print(answer)

    print("\nSOURCES USED:")
    for doc in relevant_docs:
        print(
            f"- Page: {doc.metadata.get('page', 'unknown')} | "
            f"Source: {doc.metadata.get('source', 'unknown')}"
        )
    print("=" * 80)

In [ ]:
ask_pdf("What is the core idea of a value ladder?")

In [36]:
ask_pdf("What is the core idea of a communication over landing pages?")

QUESTION:
What is the core idea of a communication over landing pages?

ANSWER:
The core idea of communication over landing pages is to tailor your message based on the type of traffic (hot, warm, or cold) arriving at your page. Each group requires special treatment and individualized communication to effectively engage them and lead to success.

SOURCES USED:
- Page: 122 | Source: dotcom-secrets.pdf
- Page: 139 | Source: dotcom-secrets.pdf
- Page: 136 | Source: dotcom-secrets.pdf


PDF File
   ↓
PyPDFLoader
   ↓
Documents
   ↓
RecursiveCharacterTextSplitter
   ↓
Chunks
   ↓
OpenAI Embeddings
   ↓
FAISS Vector Store
   ↓
Retriever
   ↓
Relevant Context
   ↓
Prompt Template
   ↓
ChatOpenAI
   ↓
Answer

Common Errors and Fixes

## Error 1: API key not found

Solution:

Check your `.env` file.

It should contain:

OPENAI_API_KEY=your_api_key_here

## Error 2: PDF not found

Solution:

Keep your PDF in the same folder as this notebook.

Or update this line:

pdf_path = Path("your_file_name.pdf")

## Error 3: Module not found

Solution:

Run the install command again:

pip install -U openai langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf python-dotenv pydantic

## Error 4: Wrong model name

Solution:

Use a valid model name in `.env`.

Example:

OPENAI_MODEL=gpt-4.1-mini

## Assinments 1: 

# 19. Student Assignments

## Assignment 1

Change the PDF and build your own PDF assistant.

Examples:

- HR policy assistant
- Resume assistant
- Legal document assistant
- Course notes assistant
- Finance report assistant

## Assignment 2

Modify the prompt so that the answer always comes in this format:

1. Short Answer
2. Detailed Explanation
3. Example
4. Source Page

## Assignment 3

Create a structured output version where the answer returns:

- answer
- confidence
- source_pages
- missing_information

## Assignment 4

Save the FAISS vector store locally and load it again without recreating embeddings every time.

so Today we learned:

- How to call OpenAI directly
- How to call OpenAI using LangChain
- How to create prompt templates
- How to generate structured output
- How to load a PDF
- How to split documents
- How to create embeddings
- How to store chunks in FAISS
- How to retrieve relevant chunks
- How to build a PDF question-answering assistant

## Final Formula

LLM alone = smart reply

LLM + LangChain + Documents = real AI application

## RAG Formula

Retrieve relevant information first.

Then generate answer using that information.